# Proyecto medio: clasificación de enfermedades en hojas de café (RoCoLe)
Dataset: [RoCoLe](https://www.kaggle.com/datasets/nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset). Tarea: clasificación (sana / roya / araña roja). Temas: preparación de datos, Hyperband, validación cruzada.

> Ejecutar desde la raíz del repo (en Colab: clonar el repo y `%cd` a él).

In [ ]:
!pip -q install -r requirements.txt
import pandas as pd, tensorflow as tf
from src import config, data_prep, generators, tuning, cross_val
tf.keras.utils.set_random_seed(config.SEED)
config.FAST_MODE = True   # False para el entrenamiento completo

## 0. Descarga e inspección
Revisar la estructura real antes de fijar columnas/etiquetas.

In [ ]:
root = config.ROOT / "data"   # dataset incluido en el repo (3 carpetas por clase)
print(root, [d.name for d in sorted(root.iterdir())])

In [ ]:
# Etiqueta = carpeta padre. Archivos tipo C10P10E1.jpg -> planta = 'C10P10' (390 plantas, 4 fotos c/u)
df = data_prep.build_dataframe(
    root,
    label_map={"coffee___healthy": "healthy", "coffee___rust": "rust",
               "coffee___red_spider_mite": "red_spider_mite"},
    group_regex=r"^(C\d+P\d+)")
df = data_prep.add_file_hash(df)
data_prep.eda(df); df.head()

## Topic 1 – Preparación
### Partición 70/15/15 sin data leakage
Se agrupa por planta/duplicados (hash) antes de cualquier augmentation o GAN, y se guardan los CSV para reproducibilidad.

In [ ]:
train, val, test = data_prep.split_dataset(df)
data_prep.assert_no_leakage(train, val, test)
pd.concat({k: v['label'].value_counts(normalize=True) for k, v in dict(train=train, val=val, test=test).items()}, axis=1)

### Data generator (carga por lotes)

In [ ]:
li = generators.make_label_index(train['label'])
ds = generators.make_dataset(train, li, training=True, augment=True)
x, y = next(iter(ds)); print(x.shape, y.shape, li)

### GAN para la clase minoritaria
Se entrena solo con imágenes de *train*; val/test siempre son reales.

In [ ]:
import numpy as np
from src.gan import CGAN
minority = train['label'].value_counts().idxmin()
sub = train.copy()
imgs = np.stack([tf.keras.utils.img_to_array(tf.keras.utils.load_img(p, target_size=(64, 64))) for p in sub['filepath']]).astype('uint8')
gan = CGAN(len(li)); gan.fit(imgs, sub['label'].map(li).values, epochs=5 if config.FAST_MODE else 200)
fake = gan.sample(li[minority], 8)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 8, figsize=(16, 2)); [a.imshow(f) or a.axis('off') for a, f in zip(ax, fake)]

Para comparar *con vs sin GAN*, guardar las imágenes sintéticas en disco, añadirlas como filas extra a `train` (nunca a val/test) y repetir el entrenamiento final.

## Topic 2 – Hyperband

In [ ]:
best = tuning.run_hyperband(train, val)
best.values

## Topic 3 – Validación cruzada (train+val; test intacto)
- **K-fold / Stratified K-fold / Hold-out / Monte Carlo**: ejecutados abajo.
- **Leave-p-out / Leave-one-out**: requieren C(n,p) / n entrenamientos; con n≈1500 y una CNN es inviable (solo se demuestra el conteo).
- **Time-series (rolling)**: no aplica, las imágenes no tienen orden temporal.

In [ ]:
from math import comb
n = len(train) + len(val); print('LOO:', n, 'entrenamientos | Leave-2-out:', comb(n, 2))
trval = pd.concat([train, val]).reset_index(drop=True)
data_prep.make_folds(trval)
cv = cross_val.run_cv(best, trval); cv

## Evaluación final (una sola vez sobre test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from src.model import BACKBONES, build_model
pre = BACKBONES[best.values['backbone']][1]; bs = best.values.get('batch_size', 32)
model = build_model(best, len(li))
model.fit(generators.make_dataset(train, li, bs, True, True, pre), validation_data=generators.make_dataset(val, li, bs, preprocess=pre),
          epochs=2 if config.FAST_MODE else 30, callbacks=[tf.keras.callbacks.EarlyStopping('val_accuracy', patience=5, restore_best_weights=True)])
te = generators.make_dataset(test, li, bs, preprocess=pre)
yp = model.predict(te).argmax(1); yt = test['label'].map(li).values
print(classification_report(yt, yp, target_names=list(li))); print(confusion_matrix(yt, yp))